In [1]:
import os, glob
import pandas as pd
# from playground_paraphraser import DATASET_MAP
from playground_consistency_analysis import get_paths
# from math_verify import parse, LatexExtractionConfig, verify
import math_verify

from llm_consistency.datasets.registry import get_dataset_spec
from llm_consistency.core.paths import ProjectPaths



# dataset = "SimpleQA"
# experiment_flag = "with_tones"
dataset = "gsm8k_test"
# dataset = "math_500_test"
# experiment_flag = "with_tones"
experiment_flag = "v00"
temperature = 0.0
question_subset="both"
paths = ProjectPaths()
rp = paths.run_paths(dataset, experiment_flag)
conf = rp.conf_suffix(temperature=temperature)
dataset_cfg = get_dataset_spec(dataset)

dataset_df = pd.read_csv(paths.dataset_file(dataset_cfg["path"]))
qcol = dataset_cfg["question_col"]

df = pd.read_csv(rp.answers_all_models_file("both", conf))
print(df.shape)

(48075, 5)


In [2]:
# Load gold dataset
dataset_df = pd.read_csv(paths.dataset_file(dataset_cfg["path"]))
qcol = dataset_cfg["question_col"]

def add_gold_columns(df):
    """Adds gold question & gold answer columns based on df['idx']."""
    df = df.copy()
    df["gold_answer"] = df["idx"].apply(lambda i: dataset_df.iloc[i]["answer"])
    df["gold_answer_parsed"] = df["idx"].apply(lambda i: math_verify.parse(f'${dataset_df.iloc[i]["answer"]}$'))
    df["answer_parsed"] = df["answer"].apply(
            lambda ans: math_verify.parse(
                ans,
                fallback_mode="no_fallback",
                extraction_config=[
                    math_verify.LatexExtractionConfig(
                        boxed_match_priority=0,
                        try_extract_without_anchor=True,
                    ),
                ]
            )
        )
    df["correct"] = df.apply(lambda row: math_verify.verify(row["gold_answer_parsed"], row["answer_parsed"]), axis=1)
    return df

new_df = add_gold_columns(df)

print("Gold columns added to both 'paraphrased_only' and 'original_only'.")
new_df


Gold columns added to both 'paraphrased_only' and 'original_only'.


,model,idx,original_question,paraphrased_question,answer,gold_answer,gold_answer_parsed,answer_parsed,correct
0,Qwen/Qwen3-8B,0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet’s ducks lay 16 eggs per day. She eats th...,Let's break the problem down step by step.\n\n...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,"[18, 18]",[18],True
1,Qwen/Qwen3-8B,0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet’s ducks produce 16 eggs daily; she consu...,Let's break the problem down step by step:\n\n...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,"[18, 18]",[18],True
2,Qwen/Qwen3-8B,0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet’s ducks produce 16 eggs daily; after eat...,Let's break the problem down step by step:\n\n...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,"[18, 18]",[18],True
3,Qwen/Qwen3-8B,0,Janet’s ducks lay 16 eggs per day. She eats th...,"Every day, Janet's ducks produce 16 eggs; afte...",Let's break the problem down step by step:\n\n...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,"[18, 18]",[18],True
4,Qwen/Qwen3-8B,0,Janet’s ducks lay 16 eggs per day. She eats th...,How much money does Janet earn each day sellin...,Let's break this down step by step:\n\n---\n\n...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,"[18, 18]",[notenough\inftyormationtodeterminetheexactamo...,False
...,...,...,...,...,...,...,...,...,...
48070,openai/gpt-oss-20b,999,A family of 6 (2 adults and 4 kids) are to div...,A family of six (2 adults and 4 kids) divides ...,analysisWe need to parse the problem: A family...,Let x be the percentage of watermelon that eac...,"[25, 25]",[25*(1/100)],False
48071,openai/gpt-oss-20b,999,A family of 6 (2 adults and 4 kids) are to div...,For a family of six with two adults and four k...,"analysisWe need to parse the problem: ""For a f...",Let x be the percentage of watermelon that eac...,"[25, 25]",[25*(1/100)],False
48072,openai/gpt-oss-20b,999,A family of 6 (2 adults and 4 kids) are to div...,"For a family of 6 with 2 adults and 4 kids, wh...","analysisWe need to parse the problem: ""For a f...",Let x be the percentage of watermelon that eac...,"[25, 25]",[25*(1/100)],False
48073,openai/gpt-oss-20b,999,A family of 6 (2 adults and 4 kids) are to div...,"In a family of 6 (2 adults and 4 children), wh...","analysisWe need to parse the problem: ""In a fa...",Let x be the percentage of watermelon that eac...,"[25, 25]",[25*(1/100)],False


In [3]:
from sympy import sympify, simplify
from sympy.core.relational import Relational, Equality
from sympy.logic.boolalg import BooleanFunction
def extract_numeric_value_(parsed):
    """Return the true numeric value from math_verify.parse output."""
    if not parsed:
        return None
    
    obj = parsed[0]

    # Convert parse node to a string so SymPy can evaluate
    expr_str = str(obj)

    # Reject equations, booleans, etc.
    if isinstance(obj, (Relational)):
        expr = obj
    else:
        try:
            expr = sympify(expr_str)
        except Exception:
            return None

    if isinstance(expr, (Relational, BooleanFunction)):
        # ---- MINIMAL PATCH FOR YOUR FAILURE CASES ----
        try:
            rhs = expr.rhs
            if rhs.is_number:
                expr = expr.rhs
        except:
            pass

    # Normal numeric/aexpr path  
    simplified = simplify(expr)
    # Normalize float→int if possible
    if simplified.is_Integer:
        return int(simplified)
    if simplified.is_Rational:
        return float(simplified) if simplified.q != 1 else int(simplified)
    if simplified.is_Float:
        iv = int(simplified)
        return iv if iv == simplified else float(simplified)

    # Fallback: string of simplified expression
    return str(simplified)

tt = new_df.copy()
tt["ans_norm"]  = tt["answer_parsed"].apply(extract_numeric_value_)
tt["gold_norm"] = tt["gold_answer_parsed"].apply(extract_numeric_value_)
tt["correct_"]  = tt["ans_norm"] == tt["gold_norm"]


In [4]:
len(tt[tt["correct"] != tt["correct_"]]) # this should be zero if we want the entropy on values to make sense

0

In [5]:
new_df = tt.copy()
print(new_df[new_df.answer_parsed.str.len() == 0].shape, new_df.shape)
new_df = new_df[new_df.answer_parsed.str.len() > 0]
print(new_df.shape)
new_df = new_df[~new_df.ans_norm.isnull()]
print(new_df.shape)
orig_df = new_df[new_df.original_question == new_df.paraphrased_question].copy()
para_df = new_df[new_df.original_question != new_df.paraphrased_question].copy()
print(orig_df.shape, para_df.shape)

(1485, 12) (48075, 12)
(46590, 12)
(46434, 12)
(4805, 12) (41629, 12)


## HERE the actual analysis begins!

In [6]:
def original_distribution(df_orig, col="correct", suffix="_orig"):
    dist = pd.get_dummies(df_orig[col])

    # Rename boolean columns
    if True in dist.columns and False in dist.columns:
        dist = dist.rename(columns={True: "correct", False: "incorrect"})

    # Add suffix
    dist.columns = dist.columns.astype(str) + suffix

    dist["idx"] = df_orig["idx"]
    dist["model"] = df_orig["model"]
    out = dist.set_index(["idx", "model"])

    return out

def paraphrased_distribution(df_para, col="correct", suffix="_para"):
    norm = (
        df_para
        .groupby(["idx", "model"])[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )

    # Rename boolean columns
    if True in norm.columns and False in norm.columns:
        norm = norm.rename(columns={True: "correct", False: "incorrect"})

    # Add suffix
    norm.columns = norm.columns.astype(str) + suffix

    return norm

import numpy as np
def iid_mismatch_metric(group, col="ans_norm", suffix=""):
    p = group[col].value_counts(normalize=True)
    iid_mismatch_prob = 1 - (p ** 2).sum()
    distinct_answers = p.size
    mode_share = p.max()
    entropy = -(p * np.log(p)).sum()

    res = {
        "iid_mismatch_prob": iid_mismatch_prob,
        "distinct": distinct_answers,
        "mode_share": mode_share,
        "entropy": entropy,
        "num_paraphrases": len(group),
        "normalized_entropy": entropy / np.log(len(group)) if len(group) > 1 else 0.0,
        # just added!
        "maj_value": p.idxmax(),
    }
    if not suffix:
        suffix = f"_{col}"
    res = {k + suffix: v for k, v in res.items()}
    return pd.Series(res)


In [7]:
orig_dist = original_distribution(orig_df, col="correct", suffix="_orig")
para_dist = paraphrased_distribution(para_df, col="correct", suffix="_para")
both_dist = paraphrased_distribution(new_df, col="correct", suffix="_both")
para_dist2 = para_df.groupby(["idx", "model"])[["correct"]].apply(iid_mismatch_metric, col="correct", suffix="_correct_para")
para_dist3 = para_df.groupby(["idx", "model"])[["ans_norm"]].apply(iid_mismatch_metric, col="ans_norm", suffix="_ans_norm_para")
both_dist2 = new_df.groupby(["idx", "model"])[["correct"]].apply(iid_mismatch_metric, col="correct", suffix="_correct_both")
both_dist3 = new_df.groupby(["idx", "model"])[["ans_norm"]].apply(iid_mismatch_metric, col="ans_norm", suffix="_ans_norm_both")
orig_values = orig_df[["idx", "model", "ans_norm"]].rename(columns={"ans_norm": "orig_value"}).set_index(["idx", "model"])
merged = (
    orig_dist
    .join(orig_values, how="inner")
    .join(para_dist, how="inner")
    .join(para_dist2, how="inner")
    .join(para_dist3, how="inner")
    .join(both_dist, how="inner")
    .join(both_dist2, how="inner")
    .join(both_dist3, how="inner")
)
aligned = merged.copy()
for suffix in ["orig", "para", "both"]:
    cols = [f"{c}_{suffix}" for c in ["correct", "incorrect"]]
    aligned[f"{suffix}_label"] = aligned[cols].idxmax(axis=1).str.replace(f"_{suffix}", "")
    # cols = [f"{c}_{suffix}" for c in ["correct", "incorrect"]]
    # aligned[f"{suffix}_label"] = df[cols].idxmax(axis=1).str.replace(f"_{suffix}", "")
aligned["match"] = aligned["orig_label"] == aligned["para_label"]
aligned["match_value"] = aligned["orig_value"] == aligned["maj_value_ans_norm_para"]
aligned["match_"] = aligned["orig_label"] == aligned["both_label"]
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None,
    'display.expand_frame_repr', False
):
    display(
        aligned[~aligned.correct_orig]
    )
aligned.shape

,,incorrect_orig,correct_orig,orig_value,incorrect_para,correct_para,iid_mismatch_prob_correct_para,distinct_correct_para,mode_share_correct_para,entropy_correct_para,num_paraphrases_correct_para,normalized_entropy_correct_para,maj_value_correct_para,iid_mismatch_prob_ans_norm_para,distinct_ans_norm_para,mode_share_ans_norm_para,entropy_ans_norm_para,num_paraphrases_ans_norm_para,normalized_entropy_ans_norm_para,maj_value_ans_norm_para,incorrect_both,correct_both,iid_mismatch_prob_correct_both,distinct_correct_both,mode_share_correct_both,entropy_correct_both,num_paraphrases_correct_both,normalized_entropy_correct_both,maj_value_correct_both,iid_mismatch_prob_ans_norm_both,distinct_ans_norm_both,mode_share_ans_norm_both,entropy_ans_norm_both,num_paraphrases_ans_norm_both,normalized_entropy_ans_norm_both,maj_value_ans_norm_both,orig_label,para_label,both_label,match,match_value,match_
idx,model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
12,Qwen/Qwen3-8B,True,False,12,0.4,0.6,0.48,2,0.6,0.673012,10,0.292285,True,0.48,2.0,0.6,0.673012,10.0,0.292285,13.0,0.454545,0.545455,0.495868,2,0.545455,0.689009,11,0.287339,True,0.495868,2.0,0.545455,0.689009,11.0,0.287339,13.0,incorrect,correct,correct,False,False,False
14,Qwen/Qwen3-8B,True,False,0.6,1.0,0.0,0.00,1,1.0,-0.000000,10,-0.000000,False,0.18,2.0,0.9,0.325083,10.0,0.141182,0.6,1.000000,0.000000,0.000000,1,1.000000,-0.000000,11,-0.000000,False,0.165289,2.0,0.909091,0.304636,11.0,0.127043,0.6,incorrect,incorrect,incorrect,True,True,True
62,Qwen/Qwen3-8B,True,False,2500,0.8,0.2,0.32,2,0.8,0.500402,10,0.217322,False,0.80,6.0,0.3,1.695743,10.0,0.736452,2500.0,0.818182,0.181818,0.297521,2,0.818182,0.474139,11,0.197731,False,0.776860,6.0,0.363636,1.641735,11.0,0.684657,2500.0,incorrect,incorrect,incorrect,True,True,True
89,Qwen/Qwen3-8B,True,False,18,0.7,0.3,0.42,2,0.7,0.610864,10,0.265295,False,0.42,2.0,0.7,0.610864,10.0,0.265295,18.0,0.727273,0.272727,0.396694,2,0.727273,0.585953,11,0.244361,False,0.396694,2.0,0.727273,0.585953,11.0,0.244361,18.0,incorrect,incorrect,incorrect,True,True,True
119,Qwen/Qwen3-8B,True,False,99076.92,1.0,0.0,0.00,1,1.0,-0.000000,10,-0.000000,False,0.68,5.0,0.5,1.359237,10.0,0.590309,99076.92,1.000000,0.000000,0.000000,1,1.000000,-0.000000,11,-0.000000,False,0.644628,5.0,0.545455,1.294545,11.0,0.539867,99076.92,incorrect,incorrect,incorrect,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,openai/gpt-oss-20b,True,False,600,0.9,0.1,0.18,2,0.9,0.325083,10,0.141182,False,0.18,2.0,0.9,0.325083,10.0,0.141182,600.0,0.909091,0.090909,0.165289,2,0.909091,0.304636,11,0.127043,False,0.165289,2.0,0.909091,0.304636,11.0,0.127043,600.0,incorrect,incorrect,incorrect,True,True,True
978,openai/gpt-oss-20b,True,False,"{100*bluecar, 60*redcar}",0.4,0.6,0.48,2,0.6,0.673012,5,0.418166,True,0.48,2.0,0.6,0.673012,5.0,0.418166,160.0,0.500000,0.500000,0.500000,2,0.500000,0.693147,6,0.386853,False,0.611111,3.0,0.500000,1.011404,6.0,0.564475,160.0,incorrect,correct,correct,False,False,False
984,openai/gpt-oss-20b,True,False,168,0.8,0.2,0.32,2,0.8,0.500402,10,0.217322,False,0.46,3.0,0.7,0.801819,10.0,0.348225,168.0,0.818182,0.181818,0.297521,2,0.818182,0.474139,11,0.197731,False,0.429752,3.0,0.727273,0.759547,11.0,0.316756,168.0,incorrect,incorrect,incorrect,True,True,True


(4805, 41)

In [8]:
# Original label frequencies
orig_dist = (
    aligned.groupby("model")["orig_label"]
    .value_counts(normalize=True)
    .unstack().fillna(0)
    # .add_prefix("orig_")
)

# Paraphrased label frequencies
para_dist = (
    aligned.groupby("model")["para_label"]
    .value_counts(normalize=True)
    .unstack().fillna(0)
    # .add_prefix("para_")
)

# Differences
label_diff = (para_dist - orig_dist)
orig_dist = orig_dist.add_suffix("_orig")
para_dist = para_dist.add_suffix("_maj_vote")
label_diff = label_diff.add_suffix("_diff")

total_counts = aligned.groupby("model").size().rename("total_samples")

agg_maj = (
    # dist
    # .join(diff_df)
    # .join()
        orig_dist
    .join(para_dist)
    .join(label_diff)
    .join(total_counts)
)

agg_maj.sort_values("correct_orig", ascending=False).round(2)


,correct_orig,incorrect_orig,correct_maj_vote,incorrect_maj_vote,correct_diff,incorrect_diff,total_samples
model,,,,,,,
openai/gpt-oss-20b,0.94,0.06,0.93,0.07,-0.01,0.01,981
Qwen/Qwen3-8B,0.93,0.07,0.92,0.08,-0.01,0.01,988
Qwen/Qwen2.5-7B-Instruct,0.91,0.09,0.89,0.11,-0.02,0.02,989
meta-llama/Llama-3.1-8B-Instruct,0.84,0.16,0.84,0.16,-0.00,0.00,944
meta-llama/Meta-Llama-3-8B-Instruct,0.77,0.23,0.75,0.25,-0.02,0.02,903


In [9]:
dist_cols = ["correct_orig","incorrect_orig","correct_para","incorrect_para"]

dist = aligned.groupby("model")[dist_cols].mean()
diff_df = aligned.assign(
    correct_diff = aligned["correct_para"] - aligned["correct_orig"].astype(float),
    incorrect_diff = aligned["incorrect_para"] - aligned["incorrect_orig"].astype(float),
).groupby("model")[["correct_diff", "incorrect_diff"]].mean()
total_counts = aligned.groupby("model").size().rename("N")
agg_dist = dist.join(diff_df).join(total_counts)
agg_dist.sort_values("correct_orig", ascending=False).round(2)

,correct_orig,incorrect_orig,correct_para,incorrect_para,correct_diff,incorrect_diff,N
model,,,,,,,
openai/gpt-oss-20b,0.94,0.06,0.88,0.12,-0.05,0.05,981
Qwen/Qwen3-8B,0.93,0.07,0.86,0.14,-0.07,0.07,988
Qwen/Qwen2.5-7B-Instruct,0.91,0.09,0.84,0.16,-0.07,0.07,989
meta-llama/Llama-3.1-8B-Instruct,0.84,0.16,0.79,0.21,-0.05,0.05,944
meta-llama/Meta-Llama-3-8B-Instruct,0.77,0.23,0.71,0.29,-0.06,0.06,903


In [10]:
merged = (
    agg_dist[["correct_orig", "correct_para", "correct_diff", "N"]]
    .join(
        agg_maj[["correct_orig", "correct_maj_vote", "correct_diff"]],
        how="inner",
        lsuffix="_dist",
        rsuffix="_mv"
    )
)
merged.round(2)

,correct_orig_dist,correct_para,correct_diff_dist,N,correct_orig_mv,correct_maj_vote,correct_diff_mv
model,,,,,,,
Qwen/Qwen2.5-7B-Instruct,0.91,0.84,-0.07,989,0.91,0.89,-0.02
Qwen/Qwen3-8B,0.93,0.86,-0.07,988,0.93,0.92,-0.01
meta-llama/Llama-3.1-8B-Instruct,0.84,0.79,-0.05,944,0.84,0.84,-0.00
meta-llama/Meta-Llama-3-8B-Instruct,0.77,0.71,-0.06,903,0.77,0.75,-0.02
openai/gpt-oss-20b,0.94,0.88,-0.05,981,0.94,0.93,-0.01


In [11]:
mismatch_stats = (
    aligned
    .assign(mismatch = ~aligned["match"])
    .groupby("model")
    .agg(
        total_samples = ("match", "size"),
        num_mismatches = ("mismatch", "sum"),
        mismatch_rate = ("mismatch", "mean"),
        iid_mismatch_correct_prob = ("iid_mismatch_prob_correct_para", "mean"),
        normalized_entropy_correct = ("normalized_entropy_correct_para", "mean"),
        iid_mismatch_ans_norm_prob = ("iid_mismatch_prob_ans_norm_para", "mean"),
        normalized_entropy_ans_norm = ("normalized_entropy_ans_norm_para", "mean"),
        mode_share_correctness = ("mode_share_correct_para", "mean"),
        mode_share_ans_norm = ("mode_share_ans_norm_para", "mean"),
    )
)
mismatch_stats.sort_values('mismatch_rate', ascending=False).round(2)
gsm_merged = (
    agg_dist[["correct_orig", "correct_para", "correct_diff", "N"]]
    .join(mismatch_stats, how="inner")
    .sort_values("correct_orig", ascending=False)
)
gsm_merged.round(2)

,correct_orig,correct_para,correct_diff,N,total_samples,num_mismatches,mismatch_rate,iid_mismatch_correct_prob,normalized_entropy_correct,iid_mismatch_ans_norm_prob,normalized_entropy_ans_norm,mode_share_correctness,mode_share_ans_norm
model,,,,,,,,,,,,,
openai/gpt-oss-20b,0.94,0.88,-0.05,981,981,37,0.04,0.10,0.07,0.11,0.09,0.93,0.92
Qwen/Qwen3-8B,0.93,0.86,-0.07,988,988,52,0.05,0.12,0.08,0.15,0.12,0.92,0.90
Qwen/Qwen2.5-7B-Instruct,0.91,0.84,-0.07,989,989,61,0.06,0.12,0.09,0.17,0.14,0.91,0.88
meta-llama/Llama-3.1-8B-Instruct,0.84,0.79,-0.05,944,944,106,0.11,0.15,0.12,0.21,0.19,0.89,0.84
meta-llama/Meta-Llama-3-8B-Instruct,0.77,0.71,-0.06,903,903,134,0.15,0.19,0.14,0.30,0.27,0.86,0.77


In [12]:
import numpy as np
import pandas as pd

def flip_stats(group: pd.DataFrame, target_label="correct") -> pd.Series:
    orig_A = group["orig_label"] == target_label
    para_A = group["para_label"] == target_label
    ND = group[f"mode_share_{target_label}_para"] < 1

    base_A = orig_A.sum()
    base_notA = (~orig_A).sum()

    def safe_ratio(num, den):
        return np.nan if den == 0 else num / den

    return pd.Series({
        "A_to_notA":   safe_ratio((orig_A & (~para_A)).sum(), base_A),
        "notA_to_A":   safe_ratio((~orig_A & para_A).sum(), base_notA),
        "A_to_ND":     safe_ratio((orig_A & ND).sum(), base_A),
        "notA_to_ND":  safe_ratio((~orig_A & ND).sum(), base_notA),
        "ND":          ND.mean(),
    })

target_label = "correct"
# 1. Base table
df_flip = aligned.groupby("model").apply(flip_stats, target_label=target_label).reset_index()

# 2. Convert flip rates to percentages
pct_cols = ["A_to_notA", "notA_to_A", "A_to_ND", "notA_to_ND", "ND"]
df_flip[pct_cols] = df_flip[pct_cols] * 100

# 3. Compute Orig Acc = P(A_orig)
orig_acc = (
    aligned.assign(correct_orig = aligned["orig_label"] == target_label)
           .groupby("model")["correct_orig"]
           .mean() * 100
)

df_flip["Orig_Acc"] = df_flip["model"].map(orig_acc)

# 4. Reorder columns: Orig Acc first
df_flip = df_flip[
    ["model", "Orig_Acc"] + [c for c in df_flip.columns if c not in ["model", "Orig_Acc"]]
]
df_flip = df_flip.sort_values("Orig_Acc", ascending=False)
df_flip.round(2)

,model,Orig_Acc,A_to_notA,notA_to_A,A_to_ND,notA_to_ND,ND
4,openai/gpt-oss-20b,93.88,2.28,26.67,29.53,51.67,30.89
1,Qwen/Qwen3-8B,92.81,3.38,29.58,34.68,53.52,36.03
0,Qwen/Qwen2.5-7B-Instruct,91.00,4.44,23.60,36.11,56.18,37.92
2,meta-llama/Llama-3.1-8B-Instruct,84.43,6.78,35.37,39.90,66.67,44.07
3,meta-llama/Meta-Llama-3-8B-Instruct,76.97,10.65,28.85,51.22,62.50,53.82


In [13]:
import numpy as np
import pandas as pd

def compute_A_any_stats(group: pd.DataFrame, target_label="correct") -> pd.Series:
    """
    Compute A_any statistics per model, fully consistent with probability definitions.
    """

    # Basic event indicators
    orig_A = group["orig_label"] == target_label
    para_A = group["para_label"] == target_label        # majority-vote paraphrase label
    A_any  = group[f"{target_label}_para"] > 0               # at least one paraphrase was correct

    # Denominators
    base_all     = len(group)
    base_origNot = (~orig_A).sum()
    base_paraNot = (~para_A).sum()

    def P(event, base):
        return np.nan if base == 0 else (event.sum() / base) * 100

    out = {
        # P(A_orig)
        "Orig_Acc": P(orig_A, base_all),

        # P(A_any) — at least one paraphrase correct
        "A_any_all": P(A_any, base_all),

        # P(A_any | orig ≠ A)
        "A_any_given_origWrong": P(A_any & (~orig_A), base_origNot),

        # P(A_any | para majority-vote ≠ A)
        "A_any_given_paraWrong": P(A_any & (~para_A), base_paraNot),
    }

    return pd.Series(out)

target_label = "correct"
# 1. Compute for all models
df_Aany = (
    aligned.groupby("model")
           .apply(compute_A_any_stats, target_label=target_label)
           .reset_index()
)

# 2. Sort by original accuracy
df_Aany = df_Aany.sort_values("Orig_Acc", ascending=False)
df_Aany.round(2)

,model,Orig_Acc,A_any_all,A_any_given_origWrong,A_any_given_paraWrong
4,openai/gpt-oss-20b,93.88,96.74,55.00,50.77
1,Qwen/Qwen3-8B,92.81,96.66,59.15,59.26
0,Qwen/Qwen2.5-7B-Instruct,91.00,95.96,60.67,62.96
2,meta-llama/Llama-3.1-8B-Instruct,84.43,95.02,72.79,68.46
3,meta-llama/Meta-Llama-3-8B-Instruct,76.97,91.69,66.83,66.22


# END